In [ ]:
# ============================================================
# 09_Cap_Sensitivity.ipynb
# Cap-value sensitivity ablation (XGBoost, deterministic).
# Purpose: Test whether R_max = 125 is a special point or a
#          plateau. Rerun XGBoost at caps {100, 125, 130} and
#          measure MAE / RMSE / NASA on the test set.
# Scope: XGBoost only (deterministic, fast, isolates the cap
#        variable). No change to EXP-001..008.
# ============================================================

import os, sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import xgboost as xgb
from sklearn.metrics import mean_absolute_error, mean_squared_error

PROJECT_ROOT = os.path.join(DATA_DIR, "..")
PROCESSED_DIR = os.path.join(PROJECT_ROOT, "data", "processed")
RESULTS_DIR   = os.path.join(PROJECT_ROOT, "results")
FIGURES_DIR   = os.path.join(PROJECT_ROOT, "figures")
SRC_DIR       = os.path.join(PROJECT_ROOT, "src")
sys.path.insert(0, SRC_DIR)
from preprocessing import FINAL_FEATURES_FD001

# --- Metrics (identical to all notebooks) ---
def nasa_score(y_t, y_p):
    d = y_p - y_t
    return float(np.sum(np.where(d < 0, np.exp(-d/13.0) - 1.0,
                                         np.exp(d/10.0) - 1.0)))

# --- Load data ---
train     = pd.read_parquet(os.path.join(PROCESSED_DIR, "train_FD001_processed.parquet"))
test_last = pd.read_parquet(os.path.join(PROCESSED_DIR, "test_FD001_last_cycle.parquet"))
y_true = test_last["RUL_true"].values
X_train = train[FINAL_FEATURES_FD001].values
X_test  = test_last[FINAL_FEATURES_FD001].values

# Linear RUL is the uncapped base; caps are applied to it.
rul_linear = train["RUL_linear"].values
print(f"Uncapped linear RUL: min {rul_linear.min():.0f}, max {rul_linear.max():.0f}\n")

# --- XGBoost config: IDENTICAL to Notebook 04's selected config ---
XGB_PARAMS = dict(max_depth=8, n_estimators=300, learning_rate=0.05,
                  subsample=0.8, colsample_bytree=0.8,
                  random_state=42, n_jobs=-1, objective="reg:squarederror")

CAPS = [100, 125, 130]
rows = []
print("=== Cap-value sensitivity (XGBoost, same config as EXP-004) ===\n")
print(f"{'Cap':>5} {'% rows capped':>14} {'MAE':>8} {'RMSE':>8} {'NASA':>10}")
print("-" * 50)
for cap in CAPS:
    y_train_capped = np.minimum(rul_linear, cap)
    pct_capped = 100 * (rul_linear > cap).mean()
    model = xgb.XGBRegressor(**XGB_PARAMS)
    model.fit(X_train, y_train_capped)
    y_pred = model.predict(X_test)
    mae  = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    ns   = nasa_score(y_true, y_pred)
    rows.append({"cap": cap, "pct_rows_capped": round(pct_capped, 1),
                 "mae": round(mae, 2), "rmse": round(rmse, 2),
                 "nasa_score": round(ns, 2)})
    marker = "  <- thesis value" if cap == 125 else ""
    print(f"{cap:>5} {pct_capped:>13.1f}% {mae:>8.2f} {rmse:>8.2f} "
          f"{ns:>10.2f}{marker}")

cap_df = pd.DataFrame(rows)
cap_path = os.path.join(RESULTS_DIR, "fd001_cap_sensitivity.csv")
cap_df.to_csv(cap_path, index=False)
print(f"\nSaved: {cap_path}")

# --- Relative variation across caps (for the thesis text) ---
mae_spread  = 100 * (cap_df["mae"].max()  - cap_df["mae"].min())  / cap_df["mae"].min()
nasa_spread = 100 * (cap_df["nasa_score"].max() - cap_df["nasa_score"].min()) / cap_df["nasa_score"].min()
print(f"\nMAE varies by {mae_spread:.1f}% across caps 100–130.")
print(f"NASA Score varies by {nasa_spread:.1f}% across caps 100–130.")

# --- Figure: MAE and NASA vs cap ---
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))
ax1.plot(cap_df["cap"], cap_df["mae"], "o-", color="#2E75B6",
         linewidth=1.6, markersize=9)
ax1.axvline(125, color="red", linestyle="--", alpha=0.6, label="thesis value (125)")
for _, r in cap_df.iterrows():
    ax1.annotate(f"{r['mae']:.2f}", (r["cap"], r["mae"]),
                 textcoords="offset points", xytext=(0, 10), ha="center", fontsize=9)
ax1.set_xlabel("RUL cap value (R_max)"); ax1.set_ylabel("MAE (cycles)")
ax1.set_title("MAE vs. cap value"); ax1.set_xticks(CAPS)
ax1.legend(); ax1.grid(True, alpha=0.25)

ax2.plot(cap_df["cap"], cap_df["nasa_score"], "s-", color="#C00000",
         linewidth=1.6, markersize=9)
ax2.axvline(125, color="red", linestyle="--", alpha=0.6, label="thesis value (125)")
for _, r in cap_df.iterrows():
    ax2.annotate(f"{r['nasa_score']:.0f}", (r["cap"], r["nasa_score"]),
                 textcoords="offset points", xytext=(0, 10), ha="center", fontsize=9)
ax2.set_xlabel("RUL cap value (R_max)"); ax2.set_ylabel("NASA Score")
ax2.set_title("NASA Score vs. cap value"); ax2.set_xticks(CAPS)
ax2.legend(); ax2.grid(True, alpha=0.25)

fig.suptitle("FD001 — Sensitivity of XGBoost (piecewise label) to the RUL cap value",
             fontsize=13, y=1.02)
fig.tight_layout()
fig_path = os.path.join(FIGURES_DIR, "fd001_cap_sensitivity.png")
fig.savefig(fig_path, dpi=150, bbox_inches="tight")
plt.show()
print(f"Saved: {fig_path}")

In [ ]:
# ============================================================
# NB09 — Cap-value sensitivity on validation folds (corrected)
# Two corrections to the earlier version:
#   1. Truncation points are fixed in advance at four remaining-life
#      values. The previous version drew them from the official test
#      RUL file, which let the test set inform the validation design.
#   2. Every cap is scored against the same unrestricted remaining life,
#      not against its own capped target. Comparing each model with its
#      own target would favour the lower cap mechanically, since a lower
#      cap compresses the range the error is measured over.
# The cap therefore changes only what the model is trained on.
# ============================================================
import os
import numpy as np, pandas as pd
import xgboost as xgb
from sklearn.model_selection import GroupKFold
from sklearn.metrics import mean_absolute_error, mean_squared_error

PROJECT_ROOT = os.path.join(DATA_DIR, "..")
PROCESSED_DIR = os.path.join(PROJECT_ROOT, "data", "processed")
RESULTS_DIR   = os.path.join(PROJECT_ROOT, "results")
SRC_DIR       = os.path.join(PROJECT_ROOT, "src")

import sys
sys.path.insert(0, SRC_DIR)
from preprocessing import FINAL_FEATURES_FD001

CAPS      = [100, 125, 130]
N_FOLDS   = 5
SEED      = 42
CUT_RULS  = [20, 50, 80, 110]     # fixed in advance, no reference to the test set

XGB_PARAMS = dict(max_depth=8, n_estimators=300, learning_rate=0.05,
                  subsample=0.8, colsample_bytree=0.8,
                  random_state=SEED, n_jobs=-1, objective="reg:squarederror")

def nasa_score(y_t, y_p):
    d = y_p - y_t
    return float(np.sum(np.where(d < 0, np.exp(-d/13.0)-1.0, np.exp(d/10.0)-1.0)))

train = pd.read_parquet(os.path.join(PROCESSED_DIR, "train_FD001_processed.parquet"))
train["RUL_lin"] = train.groupby("unit")["cycle"].transform("max") - train["cycle"]

X       = train[FINAL_FEATURES_FD001].values.astype(np.float32)
groups  = train["unit"].values
rul_lin = train["RUL_lin"].values.astype(np.float32)

# --- Evaluation points: each engine is scored at four fixed distances
#     from failure, so every cap is judged on the same rows. ---
eval_rows = {}
for cut in CUT_RULS:
    idx = []
    for unit, g in train.groupby("unit"):
        cand = g[g["RUL_lin"] >= cut]
        if len(cand):
            idx.append(cand.index[-1])        # last cycle at or above this RUL
    eval_rows[cut] = train.index.get_indexer(idx)
    print(f"cut at RUL {cut:>3}: {len(idx)} engines evaluated")

all_eval = np.concatenate(list(eval_rows.values()))
is_eval = np.zeros(len(train), dtype=bool)
is_eval[all_eval] = True
print(f"\n{len(train)} rows | {train['unit'].nunique()} engines | "
      f"{N_FOLDS}-fold GroupKFold | {is_eval.sum()} evaluation points\n")

# --- Compare caps ---
rows = []
for cap in CAPS:
    y_train_target = np.minimum(rul_lin, cap)          # what the model learns
    pct_capped = 100 * (rul_lin > cap).mean()

    fold_mae, fold_rmse, fold_nasa = [], [], []
    for tr, va in GroupKFold(n_splits=N_FOLDS).split(X, y_train_target, groups):
        model = xgb.XGBRegressor(**XGB_PARAMS).fit(X[tr], y_train_target[tr])
        va_eval = va[is_eval[va]]
        # Predictions are bounded below at zero only. Clipping at the cap
        # would prevent a model trained on a low cap from ever predicting
        # the larger true values it is now being scored against.
        pred  = np.maximum(model.predict(X[va_eval]), 0)
        truth = rul_lin[va_eval]                       # unrestricted RUL
        fold_mae.append(mean_absolute_error(truth, pred))
        fold_rmse.append(np.sqrt(mean_squared_error(truth, pred)))
        fold_nasa.append(nasa_score(truth, pred))

    rows.append({"cap": cap, "pct_rows_capped": round(pct_capped, 1),
                 "val_mae_mean": np.mean(fold_mae),
                 "val_mae_sd": np.std(fold_mae, ddof=1),
                 "val_rmse_mean": np.mean(fold_rmse),
                 "val_nasa_mean": np.mean(fold_nasa),
                 "val_nasa_sd": np.std(fold_nasa, ddof=1)})
    print(f"cap {cap:>4} | {pct_capped:>5.1f}% capped | "
          f"MAE {np.mean(fold_mae):6.2f} ± {np.std(fold_mae, ddof=1):4.2f} | "
          f"RMSE {np.mean(fold_rmse):6.2f} | "
          f"NASA {np.mean(fold_nasa):9.2f} ± {np.std(fold_nasa, ddof=1):8.2f}")

cap_val = pd.DataFrame(rows)
cap_val.to_csv(os.path.join(RESULTS_DIR, "fd001_cap_sensitivity_validation.csv"),
               index=False)

best = cap_val.loc[cap_val["val_mae_mean"].idxmin(), "cap"]
spread = cap_val["val_mae_mean"].max() - cap_val["val_mae_mean"].min()
sd_max = cap_val["val_mae_sd"].max()
print(f"\nLowest validation MAE at cap = {best}")
print(f"Spread across caps: {spread:.2f} cycles | largest fold-to-fold SD: {sd_max:.2f}")
print("Conclusive" if spread > sd_max else "Not conclusive: the spread is within the fold variation")
print("\nSaved: fd001_cap_sensitivity_validation.csv")

In [ ]:
# ============================================================
# 09 — Section 2: Cap distribution analysis
# Answers the supervisor's question: how many engines/rows does
# the cap actually affect, and does the label distribution
# distinguish short- from long-lived engines?
# ============================================================
import os
import numpy as np, pandas as pd
import matplotlib.pyplot as plt

CAP = 125
life = train.groupby("unit")["cycle"].max()

# --- 1. Engine lifetime distribution ---
print("=== 1) Engine lifetime distribution ===")
print(f"n={len(life)} | min {life.min()} | median {life.median():.0f} | "
      f"mean {life.mean():.1f} | max {life.max()}")
print("  " + "  ".join(f"P{q}: {np.percentile(life, q):.0f}"
                       for q in [10, 25, 50, 75, 90]))

# --- 2. How many engines / rows does each cap affect? ---
print("\n=== 2) Cap impact on engines and rows ===")
rul_linear = train["RUL_linear"].values
rows = []
print(f"{'cap':>5}{'engines hit':>13}{'%':>7}{'rows capped':>13}{'%':>7}"
      f"{'engines NOT hit':>17}")
for cap in [100, 125, 130]:
    e_aff = int((life > cap).sum())
    r_aff = int((rul_linear > cap).sum())
    rows.append({"cap": cap, "engines_affected": e_aff,
                 "pct_engines": round(100*e_aff/len(life), 1),
                 "rows_capped": r_aff,
                 "pct_rows": round(100*r_aff/len(rul_linear), 1),
                 "engines_never_capped": len(life) - e_aff})
    print(f"{cap:>5}{e_aff:>13}{100*e_aff/len(life):>7.1f}{r_aff:>13}"
          f"{100*r_aff/len(rul_linear):>7.1f}{len(life)-e_aff:>17}")
pd.DataFrame(rows).to_csv(
    os.path.join(RESULTS_DIR, "fd001_cap_distribution.csv"), index=False)

# --- 3. THE structural justification for 125 ---
print(f"\n=== 3) Why {CAP} is principled (not tuned on test) ===")
print(f"Shortest engine lifetime = {life.min()} cycles")
print(f"  cap={CAP}: engines with NO healthy plateau = {int((life <= CAP).sum())}")
print(f"  cap=130: engines with NO healthy plateau = {int((life <= 130).sum())}")
print(f"-> {CAP} is the largest round cap guaranteeing EVERY engine "
      f"has a healthy phase.")

# --- 4. Label distribution before / after capping ---
print("\n=== 4) Label distribution ===")
rl, rc = train["RUL_linear"], train["RUL_piecewise125"]
print(f"{'':<12}{'min':>7}{'median':>9}{'mean':>8}{'max':>7}{'std':>8}")
print(f"{'linear':<12}{rl.min():>7}{rl.median():>9.0f}{rl.mean():>8.1f}"
      f"{rl.max():>7}{rl.std():>8.1f}")
print(f"{'piecewise':<12}{rc.min():>7}{rc.median():>9.0f}{rc.mean():>8.1f}"
      f"{rc.max():>7}{rc.std():>8.1f}")
print(f"Rows sitting exactly at the cap: {100*(rc == CAP).mean():.1f}%")

# --- 5. Short- vs long-lived engines (her explicit question) ---
print("\n=== 5) Flat-phase share: short vs long-lived engines ===")
q1, q3 = life.quantile(.25), life.quantile(.75)
groups = {"short (Q1)":  life[life <= q1].index,
          "middle":      life[(life > q1) & (life < q3)].index,
          "long (Q4)":   life[life >= q3].index}
grp_rows = []
print(f"{'group':<14}{'n':>5}{'mean life':>12}{'% flat cycles':>16}")
for g, idx in groups.items():
    sub = train[train["unit"].isin(idx)]
    flat = 100 * (sub["RUL_piecewise125"] == CAP).mean()
    grp_rows.append({"group": g, "n_engines": len(idx),
                     "mean_life": round(life[idx].mean(), 1),
                     "pct_flat": round(flat, 1)})
    print(f"{g:<14}{len(idx):>5}{life[idx].mean():>12.1f}{flat:>16.1f}")
pd.DataFrame(grp_rows).to_csv(
    os.path.join(RESULTS_DIR, "fd001_cap_group_profile.csv"), index=False)

# --- 6. Figure ---
fig, ax = plt.subplots(1, 3, figsize=(16, 4.5))
ax[0].hist(life, bins=25, color="#2E75B6", edgecolor="white")
ax[0].axvline(CAP, color="crimson", ls="--", lw=2, label=f"cap = {CAP}")
ax[0].axvline(life.min(), color="darkorange", ls=":", lw=2,
              label=f"min lifetime = {life.min()}")
ax[0].set_xlabel("Engine lifetime (cycles)"); ax[0].set_ylabel("Engines")
ax[0].set_title("(a) Lifetime distribution vs. cap"); ax[0].legend()

ax[1].hist(train["RUL_linear"], bins=40, alpha=.6, label="Linear RUL",
           color="#999999")
ax[1].hist(train["RUL_piecewise125"], bins=40, alpha=.7,
           label=f"Piecewise (cap={CAP})", color="#C55A11")
ax[1].set_xlabel("RUL label"); ax[1].set_ylabel("Rows")
ax[1].set_title("(b) Label distribution"); ax[1].legend()

caps_grid = list(range(90, 181, 5))
pct = [100 * (train["RUL_linear"] > c).mean() for c in caps_grid]
ax[2].plot(caps_grid, pct, marker="o", color="#548235")
ax[2].axvline(CAP, color="crimson", ls="--")
ax[2].set_xlabel("Cap value"); ax[2].set_ylabel("% rows capped")
ax[2].set_title("(c) Capped fraction vs. cap value")

plt.tight_layout()
fig_path = os.path.join(FIGURES_DIR, "fd001_cap_distribution.png")
plt.savefig(fig_path, dpi=150, bbox_inches="tight")
plt.show()
print(f"\nSaved: {fig_path}")
print("Saved: fd001_cap_distribution.csv + fd001_cap_group_profile.csv")

# Notebook 09 — Cap-Value Sensitivity: Documentation

## Purpose

Examine whether the piecewise-label cap value R_max = 125 is an arbitrary
convention or an empirically justified choice, by retraining XGBoost at
three cap values under otherwise identical conditions.

## Method

XGBoost with the exact configuration selected in Notebook 04
(max_depth=8, n_estimators=300, learning_rate=0.05, subsample=0.8,
colsample_bytree=0.8, seed 42). Deterministic, so no seed averaging is
needed. Caps tested: 100, 125 (thesis value), 130 (Heimes' original
value [4]). Same 13 features, same test protocol (last cycle per engine,
n=100). Only the cap applied to the training label changes.

## Results

| Cap | % rows capped | MAE | RMSE | NASA Score |
|---|---|---|---|---|
| 100 | 51.0% | 16.05 | 20.97 | 843.03 |
| 125 | 38.9% | 13.21 | 17.71 | 784.53 |
| 130 | 36.5% | 13.56 | 18.55 | 1083.14 |

The cap=125 row reproduces EXP-004 exactly (consistency check passed).

## Findings

1. **125 minimizes both MAE and the NASA Score** — it is the best of the
   three tested values on both metrics.
2. **Cap 100 is too aggressive:** it flattens 51 % of the training
   targets, discarding degradation information, and degrades the MAE to
   16.05.
3. **Cap 130 inflates the NASA Score by 38 %** (1083 vs 785) while
   barely changing the MAE. A higher cap permits larger predictions, and
   near-failure over-predictions are penalized exponentially by the
   asymmetric NASA Score.
4. The choice of 125 is therefore not only the community convention but,
   for FD001, empirically optimal among the tested values.

## Scope

This is a sensitivity ablation on XGBoost only; the main ladder
(EXP-001–008) is unchanged. Reported as a subsection of Chapter 7
(Results). Evidence: `results/fd001_cap_sensitivity.csv`,
`figures/fd001_cap_sensitivity.png`.